In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dataguruji/redrobe/job_description.docx
/kaggle/input/datasets/dataguruji/redrobe/candidates.jsonl
/kaggle/input/datasets/dataguruji/redrobe/candidate_schema.json


In [7]:
# ===========================================
# Install Only Missing Packages
# ===========================================

try:
    import yake
except ImportError:
    !pip install -q yake

try:
    import rank_bm25
except ImportError:
    !pip install -q rank-bm25

import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except:
    !python -m spacy download en_core_web_sm

In [8]:
!pip install -q python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.3 MB/s eta 0:00:00


In [9]:
# ===========================================
# Imports
# ===========================================

import os
import re
import json
import random
import pickle
import warnings

from pathlib import Path
from docx import Document

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

import spacy
import yake
import networkx as nx

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

In [10]:
# ===========================================
# Configuration
# ===========================================

class CFG:

    RANDOM_STATE = 42

    EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

    SPACY_MODEL = "en_core_web_sm"

    TOP_KEYWORDS = 30

    TOP_SENTENCES = 15

    OUTPUT_DIR = "/kaggle/working"

CFG = CFG()

In [11]:
# ===========================================
# Seed Everything
# ===========================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

seed_everything(CFG.RANDOM_STATE)

In [12]:
# ===========================================
# Load NLP Models
# ===========================================

print("Loading spaCy...")

nlp = spacy.load(CFG.SPACY_MODEL)

print("Loading Embedding Model...")

embedding_model = SentenceTransformer(
    CFG.EMBEDDING_MODEL
)

print("Done!")

Loading spaCy...


Loading Embedding Model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Done!


In [13]:
# ===========================================
# Create Output Folder
# ===========================================

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

print(CFG.OUTPUT_DIR)

/kaggle/working


In [14]:
# ===========================================
# JD Loader
# ===========================================

class JDLoader:

    @staticmethod

    def load(path):

        extension = Path(path).suffix.lower()

        if extension == ".docx":

            doc = Document(path)

            text = "\n".join(

                paragraph.text.strip()

                for paragraph in doc.paragraphs

                if paragraph.text.strip()

            )

            return text

        elif extension == ".md":

            with open(path, "r", encoding="utf-8") as f:

                return f.read()

        elif extension == ".txt":

            with open(path, "r", encoding="utf-8") as f:

                return f.read()

        else:

            raise ValueError(
                f"Unsupported File Type: {extension}"
            )

In [15]:
# ===========================================
# Load Job Description
# ===========================================

JD_PATH = "/kaggle/input/datasets/dataguruji/redrobe/job_description.docx"

jd_text = JDLoader.load(JD_PATH)

print("="*80)

print(jd_text[:2000])

print("="*80)

Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)
Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before.
If you've spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn't it.
If you've spent your career bouncing between early-stage startups and you want to "just code" without having to think a

In [16]:
# ===========================================
# Text Cleaning
# ===========================================

def clean_text(text):

    text = text.replace("\xa0", " ")

    text = re.sub(r"\s+", " ", text)

    text = re.sub(r"\n+", "\n", text)

    return text.strip()


jd_text = clean_text(jd_text)

In [17]:
# ===========================================
# Basic Statistics
# ===========================================

jd_doc = nlp(jd_text)

sentences = list(jd_doc.sents)

tokens = [token.text for token in jd_doc]

print("="*60)

print("JOB DESCRIPTION STATISTICS")

print("="*60)

print(f"Characters : {len(jd_text):,}")

print(f"Words      : {len(jd_text.split()):,}")

print(f"Tokens     : {len(tokens):,}")

print(f"Sentences  : {len(sentences):,}")

print("="*60)

JOB DESCRIPTION STATISTICS
Characters : 9,564
Words      : 1,514
Tokens     : 1,989
Sentences  : 92


In [18]:
print(os.path.exists(JD_PATH))
print(len(jd_text))

True
9564


In [19]:
# ============================================================
# Process Job Description using spaCy
# ============================================================

jd_doc = nlp(jd_text)

print("spaCy processing completed!")

spaCy processing completed!


In [20]:
# ============================================================
# Sentence Segmentation
# ============================================================

sentences = [sent.text.strip() for sent in jd_doc.sents if sent.text.strip()]

print(f"Total Sentences: {len(sentences)}")

print("\nFirst 5 Sentences:\n")

for i, sent in enumerate(sentences[:5], start=1):
    print(f"{i}. {sent}\n")

Total Sentences: 92

First 5 Sentences:

1. Job Description: Senior AI Engineer — Founding Team Company: Redrob AI (Series A AI-native talent intelligence platform) Location: Pune/Noida, India (Hybrid — flexible cadence)

2. | Open to relocation candidates from Tier-1 Indian cities Employment Type: Full-time Experience Required:

3. 5–9 years (see "what we mean by this" below) Let's be honest about this role We're going to write this JD differently from most.

4. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch.

5. This is the kind of role where the JD changes every six months because the company changes every six months.



In [21]:
# ============================================================
# Token Analysis
# ============================================================

tokens = [token.text for token in jd_doc]

print(f"Total Tokens : {len(tokens)}")
print(tokens[:50])

Total Tokens : 1989
['Job', 'Description', ':', 'Senior', 'AI', 'Engineer', '—', 'Founding', 'Team', 'Company', ':', 'Redrob', 'AI', '(', 'Series', 'A', 'AI', '-', 'native', 'talent', 'intelligence', 'platform', ')', 'Location', ':', 'Pune', '/', 'Noida', ',', 'India', '(', 'Hybrid', '—', 'flexible', 'cadence', ')', '|', 'Open', 'to', 'relocation', 'candidates', 'from', 'Tier-1', 'Indian', 'cities', 'Employment', 'Type', ':', 'Full', '-']


In [22]:
# ============================================================
# Lemmatization
# ============================================================

lemmas = [

    token.lemma_.lower()

    for token in jd_doc

    if not token.is_stop
    and not token.is_punct
]

print(f"Total Lemmas : {len(lemmas)}")

print(lemmas[:100])

Total Lemmas : 872
['job', 'description', 'senior', 'ai', 'engineer', 'found', 'team', 'company', 'redrob', 'ai', 'series', 'ai', 'native', 'talent', 'intelligence', 'platform', 'location', 'pune', 'noida', 'india', 'hybrid', 'flexible', 'cadence', '|', 'open', 'relocation', 'candidate', 'tier-1', 'indian', 'city', 'employment', 'type', 'time', 'experience', 'required', '5–9', 'year', 'mean', 'let', 'honest', 'role', 'go', 'write', 'jd', 'differently', 'series', 'company', 'raise', 'round', 'build', 'new', 'ai', 'engineering', 'org', 'scratch', 'kind', 'role', 'jd', 'change', 'month', 'company', 'change', 'month', 'instead', 'pretend', 'fix', 'checklist', 'go', 'tell', 'actually', 'need', 'get', 'wrong', 'spend', 'career', 'google', 'meta', 'want', 'scope', 'role', 'define', 'ladder', 'spend', 'career', 'bounce', 'early', 'stage', 'startup', 'want', 'code', 'have', 'think', 'product', 'recruiter', 'workflow', 'eval', 'framework', 'need', 'simultaneously', 'comfortable']


In [23]:
# ============================================================
# POS Tag Distribution
# ============================================================

from collections import Counter

pos_counts = Counter(token.pos_ for token in jd_doc)

pos_df = pd.DataFrame(
    pos_counts.items(),
    columns=["POS", "Count"]
).sort_values("Count", ascending=False)

display(pos_df)

,POS,Count
2,NOUN,391
1,PUNCT,338
3,VERB,206
9,PRON,181
7,ADP,138
4,ADJ,127
11,AUX,109
0,PROPN,105
12,DET,97
10,ADV,74


In [24]:
# ============================================================
# Named Entity Recognition
# ============================================================

entities = []

for ent in jd_doc.ents:

    entities.append({

        "Entity": ent.text,

        "Label": ent.label_
    })

entities_df = pd.DataFrame(entities)

display(entities_df.head(20))

,Entity,Label
0,Pune/Noida,ORG
1,India,GPE
2,Indian,NORP
3,JD,GPE
4,AI Engineering,ORG
5,JD,PERSON
6,every six months,DATE
7,Google,ORG
8,Meta,PERSON
9,two,CARDINAL


In [25]:
# ============================================================
# Noun Chunk Extraction
# ============================================================

noun_chunks = list(set(

    chunk.text.strip().lower()

    for chunk in jd_doc.noun_chunks

))

print(f"Total Noun Chunks : {len(noun_chunks)}")

noun_chunks[:50]

Total Noun Chunks : 298


['candidate-jd matching',
 'the gap',
 '"architecture',
 '30+ day notice candidates',
 'the underlying ml',
 'the job market',
 'google',
 'delhi ncr',
 'this stage',
 'the evaluation infrastructure',
 'this kind',
 'behavioral signals',
 'recommendation system',
 'product',
 'redrob platform',
 'bge',
 'people',
 'qdrant',
 'the right answer',
 '6 months',
 'we',
 'a candidate',
 'i',
 'a defined ladder',
 'eval frameworks',
 'a series a company',
 'the participants',
 'the intelligent candidate discovery',
 'your "ai experience',
 'both modes',
 'your first 90 days',
 'they',
 'things',
 'recruiter-feedback loops',
 'ndcg',
 'a requirement',
 '(when to fine-tune',
 'paper',
 'a fixed checklist',
 'a new ai engineering org',
 'a ranking system',
 'hires',
 'a recommendation system',
 'logistics location',
 'culture-fit matters',
 'a working ranker',
 'the bar',
 'mrr',
 '— flexible cadence',
 'a/b test interpretation']

In [26]:
# ============================================================
# Dependency Parsing Sample
# ============================================================

dependency_df = pd.DataFrame({

    "Token": [token.text for token in jd_doc],

    "Dependency": [token.dep_ for token in jd_doc],

    "Head": [token.head.text for token in jd_doc],

    "POS": [token.pos_ for token in jd_doc]

})

dependency_df.head(30)

,Token,Dependency,Head,POS
0,Job,compound,Description,PROPN
1,Description,ROOT,Description,PROPN
2,:,punct,Description,PUNCT
3,Senior,amod,Engineer,PROPN
4,AI,compound,Engineer,PROPN
5,Engineer,compound,Company,NOUN
6,—,punct,Company,PUNCT
7,Founding,compound,Company,VERB
8,Team,compound,Company,PROPN
9,Company,appos,Description,PROPN


In [27]:
# ============================================================
# Final NLP Statistics
# ============================================================

print("="*60)

print("JOB DESCRIPTION NLP SUMMARY")

print("="*60)

print("Characters      :", len(jd_text))

print("Words           :", len(jd_text.split()))

print("Sentences       :", len(sentences))

print("Tokens          :", len(tokens))

print("Lemmas          :", len(lemmas))

print("Entities        :", len(entities))

print("Noun Chunks     :", len(noun_chunks))

print("="*60)

JOB DESCRIPTION NLP SUMMARY
Characters      : 9564
Words           : 1514
Sentences       : 92
Tokens          : 1989
Lemmas          : 872
Entities        : 88
Noun Chunks     : 298


In [36]:
# ============================================================
# Generate Sentence Embeddings
# ============================================================

sentence_embeddings = embedding_model.encode(
    sentences,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(sentence_embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(92, 768)


In [37]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(sentence_embeddings)

print(similarity_matrix.shape)

(92, 92)


In [38]:
graph = nx.from_numpy_array(similarity_matrix)

sentence_scores = nx.pagerank(graph)

ranked_sentences = sorted(
    (
        (sentence_scores[i], sentences[i])
        for i in range(len(sentences))
    ),
    reverse=True
)

In [39]:
from sklearn.cluster import AgglomerativeClustering

NUM_CLUSTERS = 4

cluster_model = AgglomerativeClustering(
    n_clusters=NUM_CLUSTERS
)

cluster_labels = cluster_model.fit_predict(
    sentence_embeddings
)

In [40]:
cluster_df = pd.DataFrame({

    "Sentence": sentences,

    "Cluster": cluster_labels

})

cluster_df.head()

,Sentence,Cluster
0,Job Description: Senior AI Engineer — Founding...,1
1,| Open to relocation candidates from Tier-1 In...,2
2,"5–9 years (see ""what we mean by this"" below) L...",0
3,We're a Series A company that just raised our ...,1
4,This is the kind of role where the JD changes ...,0


In [41]:
for cluster in sorted(cluster_df.Cluster.unique()):

    print("="*80)

    print(f"CLUSTER {cluster}")

    print("="*80)

    display(
        cluster_df[
            cluster_df.Cluster==cluster
        ][["Sentence"]]
    )

CLUSTER 0


,Sentence
2,"5–9 years (see ""what we mean by this"" below) L..."
4,This is the kind of role where the JD changes ...
5,So instead of pretending we have a fixed check...
10,These are not contradictory in real life.
11,They feel contradictory because of how enginee...
12,We need both modes available in the same perso...
15,"In practical terms, your first 90 days will pr..."
20,"Beyond that, you'll be driving the long-term a..."
21,"What we mean by ""5-9 years"" This is a range, n..."
23,We've used 5-9 because it's roughly where peop...


CLUSTER 1


,Sentence
0,Job Description: Senior AI Engineer — Founding...
3,We're a Series A company that just raised our ...
8,We need someone who is simultaneously comforta...
9,Scrappy product-engineering attitude — willing...
13,What you'd actually be doing The high-level ma...
14,"That means the ranking, retrieval, and matchin..."
16,Identify the 3-4 highest-leverage things to fix.
17,Weeks 4-8: Ship a v2 ranking system that demon...
18,"This will involve embeddings, hybrid retrieval..."
19,Weeks 9-12: Set up the evaluation infrastructu...


CLUSTER 2


,Sentence
1,| Open to relocation candidates from Tier-1 In...
59,"On location, comp, and logistics Location: Pun..."
60,We have offices in Noida and Pune(mostly used ...
62,"Candidates in Hyderabad, Pune, Mumbai, Delhi N..."
78,Located in or willing to relocate to Noida or ...


CLUSTER 3


,Sentence
6,If you've spent your career at Google or Meta ...
7,If you've spent your career bouncing between e...
22,"Some people hit ""senior engineer"" judgment at ..."
24,"That said, here are the disqualifiers we actua..."
27,"If your ""AI experience"" consists primarily of ..."
29,If you are a senior engineer who hasn't writte...
31,The skills inventory (please read carefully)
32,Most JDs list 20 skills and you're supposed to...
42,If you've never thought about how to evaluate ...
44,"Prior exposure to HR-tech, recruiting tech, or..."


In [42]:
cluster_centroids = {}

for cluster in range(NUM_CLUSTERS):

    idx = np.where(cluster_labels==cluster)[0]

    cluster_centroids[cluster] = sentence_embeddings[idx].mean(axis=0)

In [43]:
from sklearn.metrics.pairwise import cosine_similarity

representatives = {}

for cluster in range(NUM_CLUSTERS):

    idx = np.where(cluster_labels==cluster)[0]

    sims = cosine_similarity(

        sentence_embeddings[idx],

        cluster_centroids[cluster].reshape(1,-1)

    ).flatten()

    representatives[cluster] = sentences[
        idx[np.argmax(sims)]
    ]

In [44]:
for cluster,sentence in representatives.items():

    print("="*80)

    print(f"Cluster {cluster}")

    print(sentence)

    print()

Cluster 0
We're going to do this differently.

Cluster 1
This will involve embeddings, hybrid retrieval, and probably some LLM-based re-ranking, but the architecture is your call.

Cluster 2
Located in or willing to relocate to Noida or Pune.

Cluster 3
If you've spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn't it.



In [45]:
semantic_documents = {}

for cluster in range(NUM_CLUSTERS):

    idx = np.where(cluster_labels==cluster)[0]

    semantic_documents[f"cluster_{cluster}"] = "\n".join(

        sentences[i]

        for i in idx

    )

In [46]:
cluster_embeddings = {}

for name,text in semantic_documents.items():

    cluster_embeddings[name] = embedding_model.encode(

        text,

        normalize_embeddings=True

    )

In [47]:
!pip install -q umap-learn hdbscan

In [48]:
import umap.umap_ as umap
import hdbscan

2026-07-01 16:19:39.192826: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782922779.363614      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782922779.414527      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782922779.823001      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782922779.823039      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782922779.823043      58 computation_placer.cc:177] computation placer alr

In [49]:
# ==========================================================
# UMAP Projection
# ==========================================================

umap_model = umap.UMAP(
    n_neighbors=10,
    n_components=10,
    min_dist=0.05,
    metric="cosine",
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(
    sentence_embeddings
)

print(reduced_embeddings.shape)

(92, 10)


In [50]:
# ==========================================================
# HDBSCAN
# ==========================================================

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=4,
    metric="euclidean",
    prediction_data=True
)

cluster_labels = clusterer.fit_predict(
    reduced_embeddings
)

cluster_df = pd.DataFrame({
    "Sentence": sentences,
    "Cluster": cluster_labels
})

cluster_df.head()

,Sentence,Cluster
0,Job Description: Senior AI Engineer — Founding...,-1
1,| Open to relocation candidates from Tier-1 In...,3
2,"5–9 years (see ""what we mean by this"" below) L...",4
3,We're a Series A company that just raised our ...,-1
4,This is the kind of role where the JD changes ...,5


In [51]:
print("Clusters Found :",
      len(set(cluster_labels))-(
          1 if -1 in cluster_labels else 0
      )
)

print("Noise Sentences :",
      np.sum(cluster_labels==-1))

Clusters Found : 6
Noise Sentences : 28


In [52]:
clusters = sorted(
    c
    for c in cluster_df.Cluster.unique()
    if c!=-1
)

for cluster in clusters:

    print("="*100)

    print(f"SEMANTIC CLUSTER {cluster}")

    print("="*100)

    display(
        cluster_df[
            cluster_df.Cluster==cluster
        ][["Sentence"]]
    )

SEMANTIC CLUSTER 0


,Sentence
5,So instead of pretending we have a fixed check...
10,These are not contradictory in real life.
11,They feel contradictory because of how enginee...
12,We need both modes available in the same perso...
25,We are explicit about this.
26,We've tried it twice and it didn't work for ei...
33,We're going to do this differently.
40,"Yes really, we care about code quality."
50,but it's not what we need.
51,"We need people who think about systems, not fr..."


SEMANTIC CLUSTER 1


,Sentence
16,Identify the 3-4 highest-leverage things to fix.
31,The skills inventory (please read carefully)
48,Framework enthusiasts.
49,If your GitHub is full of LangChain tutorials ...
68,Skills are teachable; the rest mostly isn't.


SEMANTIC CLUSTER 2


...

,Sentence
1,| Open to relocation candidates from Tier-1 In...
59,"On location, comp, and logistics Location: Pun..."
60,We have offices in Noida and Pune(mostly used ...
62,"Candidates in Hyderabad, Pune, Mumbai, Delhi N..."
63,"Outside India: case-by-case, but we don't spon..."
78,Located in or willing to relocate to Noida or ...


SEMANTIC CLUSTER 4


,Sentence
2,"5–9 years (see ""what we mean by this"" below) L..."
15,"In practical terms, your first 90 days will pr..."
21,"What we mean by ""5-9 years"" This is a range, n..."
47,We need someone who plans to be here for 3+ ye...
64,Notice period: We'd love sub-30-day notice.
65,We can buy out up to 30 days.
66,30+ day notice candidates are still in scope b...


SEMANTIC CLUSTER 5


,Sentence
4,This is the kind of role where the JD changes ...
6,If you've spent your career at Google or Meta ...
7,If you've spent your career bouncing between e...
22,"Some people hit ""senior engineer"" judgment at ..."
24,"That said, here are the disqualifiers we actua..."
29,If you are a senior engineer who hasn't writte...
30,This role writes code.
42,If you've never thought about how to evaluate ...
46,If your career trajectory shows you optimizing...
57,People whose work has been entirely on closed-...


In [53]:
semantic_clusters = {}

cluster_embeddings = {}

for cluster in clusters:

    cluster_sentences = list(

        cluster_df[
            cluster_df.Cluster==cluster
        ]["Sentence"]

    )

    document = "\n".join(cluster_sentences)

    semantic_clusters[cluster]=document

    cluster_embeddings[cluster]=embedding_model.encode(
        document,
        normalize_embeddings=True
    )

In [54]:
from sklearn.metrics.pairwise import cosine_similarity

cluster_summary = {}

for cluster in clusters:

    idx=np.where(cluster_labels==cluster)[0]

    centroid=sentence_embeddings[idx].mean(axis=0)

    sims=cosine_similarity(

        sentence_embeddings[idx],

        centroid.reshape(1,-1)

    ).flatten()

    representative=sentences[
        idx[np.argmax(sims)]
    ]

    cluster_summary[cluster]=representative

In [55]:
for cluster in clusters:

    print("="*100)

    print("Cluster",cluster)

    print()

    print("Representative Sentence")

    print(cluster_summary[cluster])

    print()

    print("Number of Sentences")

    print(
        len(
            semantic_clusters[cluster].split(".")
        )
    )

Cluster 0

Representative Sentence
We're going to do this differently.

Number of Sentences
20
Cluster 1

Representative Sentence
The skills inventory (please read carefully)

Number of Sentences
4
Cluster 2

Representative Sentence
This will involve embeddings, hybrid retrieval, and probably some LLM-based re-ranking, but the architecture is your call.

Number of Sentences
14
Cluster 3

Representative Sentence
Located in or willing to relocate to Noida or Pune.

Number of Sentences
6
Cluster 4

Representative Sentence
Notice period: We'd love sub-30-day notice.

Number of Sentences
8
Cluster 5

Representative Sentence
If you are a senior engineer who hasn't written production code in the last 18 months because you've moved into "architecture" or "tech lead" roles — we will probably not move forward.

Number of Sentences
14


In [56]:
import pickle

jd_profile = {

    "keywords": keyword_df,

    "important_sentences": important_df,

    "clusters": semantic_clusters,

    "cluster_summary": cluster_summary,

    "cluster_embeddings": cluster_embeddings,

    "sentence_embeddings": sentence_embeddings

}

with open(
    "/kaggle/working/jd_semantic_profile.pkl",
    "wb"
) as f:

    pickle.dump(jd_profile,f)

print("Saved successfully.")

Saved successfully.


In [57]:
# ==========================================================
# BM25 Representation
# ==========================================================

from rank_bm25 import BM25Okapi

tokenized_sentences = [
    sentence.lower().split()
    for sentence in sentences
]

bm25 = BM25Okapi(tokenized_sentences)

print("BM25 Index Built")

BM25 Index Built


In [58]:
from collections import Counter

word_freq = Counter()

for sentence in tokenized_sentences:
    word_freq.update(sentence)

word_df = (
    pd.DataFrame(
        word_freq.items(),
        columns=["Word","Frequency"]
    )
    .sort_values("Frequency", ascending=False)
)

display(word_df.head(30))

,Word,Frequency
71,the,44
40,we,37
24,to,33
11,a,28
138,in,23
64,and,22
49,this,19
168,not,17
73,of,16
96,if,16


In [59]:
noun_chunk_df = pd.DataFrame({
    "NounChunk": noun_chunks
})

display(noun_chunk_df.head(50))

,NounChunk
0,candidate-jd matching
1,the gap
2,"""architecture"
3,30+ day notice candidates
4,the underlying ml
5,the job market
6,google
7,delhi ncr
8,this stage
9,the evaluation infrastructure


In [60]:
entity_df = pd.DataFrame(entities)

display(entity_df)

,Entity,Label
0,Pune/Noida,ORG
1,India,GPE
2,Indian,NORP
3,JD,GPE
4,AI Engineering,ORG
...,...,...
83,5,CARDINAL
84,RAG,WORK_OF_ART
85,AI,GPE
86,6 months,DATE


In [61]:
jd_hybrid_profile = {

    "raw_text": jd_text,

    "sentences": sentences,

    "keywords": keyword_df,

    "important_sentences": important_df,

    "noun_chunks": noun_chunks,

    "entities": entities,

    "clusters": semantic_clusters,

    "cluster_summary": cluster_summary,

    "cluster_embeddings": cluster_embeddings,

    "bm25": bm25
}

In [62]:
import pickle

with open(
    "/kaggle/working/jd_hybrid_profile.pkl",
    "wb"
) as f:

    pickle.dump(
        jd_hybrid_profile,
        f
    )

print("Hybrid JD Profile Saved")

Hybrid JD Profile Saved
